In [1]:
import os
import numpy as np
import csv

In [3]:
import pandas as pd
batting = pd.read_csv(r'D:\ED\FIFS_IPL\data\mbm\ganjoo_ipl_mbm_bat_final.csv')

In [7]:
batting.sample()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
4367,2018-04-23,1136582,Punjab Kings,Delhi Capitals,1,459508,Andrew Tye,3,9,0,0,33.33,True,bowled,Trent Boult,Trent Boult,Feroz Shah Kotla


In [ ]:
batting['boundaries'] = batting_cleaned['fours'] + batting_cleaned['sixes']
batting_cleaned.boundaries.fillna(0, inplace=True)
batting_cleaned.drop(columns=['fours', 'sixes'], inplace=True)

# Merge batting_cleaned with matches_cleaned to get the match dates
batting_cleaned = batting_cleaned.merge(matches_cleaned[['Match ID', 'Match Date']], on='Match ID', how='left')

# Convert 'Match Date' to datetime format
batting_cleaned['Match Date'] = pd.to_datetime(batting_cleaned['Match Date'])

# Sort data by batsman and match date
batting_cleaned.sort_values(by=['batsman', 'Match Date'], inplace=True)

# Ensure numeric types
batting_cleaned['runs'] = pd.to_numeric(batting_cleaned['runs'], errors='coerce').fillna(0)
batting_cleaned['isOut'] = pd.to_numeric(batting_cleaned['isOut'], errors='coerce').fillna(0)
batting_cleaned['balls'] = pd.to_numeric(batting_cleaned['balls'], errors='coerce').fillna(0)

# Compute cumulative runs and dismissals
batting_cleaned['cumulative_runs'] = (
    batting_cleaned.groupby('batsman')['runs']
    .cumsum().shift()
)

batting_cleaned['cumulative_dismissals'] = (
    batting_cleaned.groupby('batsman')['isOut']
    .cumsum().shift()
)

batting_cleaned['cumulative_matches'] = (
    batting_cleaned.groupby('batsman')
    ['batsman'].cumcount() # Count rows within each batsman's group
)

batting_cleaned['cumulative_balls'] = (
    batting_cleaned.groupby('batsman')['balls']
    .cumsum().shift()
)

batting_cleaned['cumulative_boundaries'] = (
    batting_cleaned.groupby('batsman')['boundaries']
    .cumsum().shift()
)

# Compute career batting average before each match
batting_cleaned['batter_avg'] = (
    batting_cleaned['cumulative_runs'] / batting_cleaned['cumulative_dismissals'].replace(0, pd.NA)
)

# Compute Runs per match before each match
batting_cleaned['batter_rpm'] = (
    batting_cleaned['cumulative_runs'] / batting_cleaned['cumulative_matches']
)

# Compute career strike rate before each match
batting_cleaned['batter_sr'] = (
    (batting_cleaned['cumulative_runs'] * 100) /
    batting_cleaned['cumulative_balls'].replace(0, pd.NA)
)

# Compute boundaries per match before each match
batting_cleaned['batter_bpm'] = (
    batting_cleaned['cumulative_boundaries'] / batting_cleaned['cumulative_matches']
)

# Ensure first match has NaN (null) values
batting_cleaned.loc[
    batting_cleaned.groupby('batsman').head(1).index, 'batter_avg'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby('batsman').head(1).index, 'batter_rpm'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby('batsman').head(1).index, 'batter_sr'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby('batsman').head(1).index, 'batter_bpm'
] = pd.NA

# Drop unnecessary columns
batting_cleaned.drop(columns=['cumulative_runs', 'cumulative_dismissals', 'cumulative_matches', 'cumulative_boundaries', 'cumulative_balls'], inplace=True)
     